# Fraud Detection – Final Model Export

Objective:
- Train final XGBoost model
- Save trained model
- Save threshold
- Prepare for Flask deployment

In [30]:
import joblib
import numpy as np
from xgboost import XGBClassifier

In [31]:
X_train = joblib.load("../data/X_train_processed.pkl")
y_train = joblib.load("../data/y_train.pkl")
encoder = joblib.load("../models/encoder.pkl")
scaler = joblib.load("../models/scaler.pkl")
feature_names = joblib.load("../models/feature_names.pkl")

In [32]:
fraud = np.sum(y_train == 1)
non_fraud = np.sum(y_train == 0)

scale_pos_weight = non_fraud / fraud
print("Scale_pos_weight:", scale_pos_weight)

Scale_pos_weight: 81.5892361111111


In [33]:
final_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

final_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [34]:
joblib.dump(final_model, "../models/xgboost_model.pkl")

['../models/xgboost_model.pkl']

In [35]:
final_threshold = 0.8

joblib.dump(final_threshold, "../models/threshold.pkl")

['../models/threshold.pkl']

In [36]:
loaded_model = joblib.load("../models/xgboost_model.pkl")
print("Model loaded successfully.")

Model loaded successfully.


In [37]:
sample = X_train[:5]

probs = loaded_model.predict_proba(sample)[:,1]
print("Sample probabilities:", probs)

Sample probabilities: [4.1851432e-05 4.3200147e-05 5.2973948e-05 3.7519192e-05 3.5508678e-05]


## Pipeline


In [38]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Load dataset
df = pd.read_csv("../data/banksim.csv")

X = df[["step","amount","age","gender","merchant","category"]]
y = df["fraud"]

# SAME train-test split as before
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Compute imbalance weight
fraud = np.sum(y_train == 1)
non_fraud = np.sum(y_train == 0)
scale_pos_weight = non_fraud / fraud

# Define model
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["step","amount"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["age","gender","merchant","category"])
    ]
)

# Pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# Fit ONLY on training data
pipeline.fit(X_train, y_train)

# Save
joblib.dump(pipeline, "../models/full_pipeline.pkl")

['../models/full_pipeline.pkl']

In [39]:
# Get fraud cases with highest predicted probability
pipeline = joblib.load("../models/full_pipeline.pkl")

fraud_df = df[df["fraud"] == 1].copy()

fraud_df["prob"] = pipeline.predict_proba(
    fraud_df[["step","amount","age","gender","merchant","category"]]
)[:,1]

fraud_df.sort_values("prob", ascending=False).head(5)

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud,prob
504926,155,'C1372889664','5','F','28007','M1294758098','28007','es_leisure',474.48,1,0.999972
592501,179,'C137688853','4','M','28007','M3697346','28007','es_leisure',483.10,1,0.999971
338596,109,'C611182051','2','F','28007','M1294758098','28007','es_leisure',592.03,1,0.999965
311888,102,'C1978250683','3','F','28007','M3697346','28007','es_leisure',563.84,1,0.999963
308829,101,'C1994178184','4','M','28007','M3697346','28007','es_leisure',476.68,1,0.999962


# Final System Summary

1. XGBoost selected as final model.
2. scale_pos_weight used to handle imbalance.
3. Threshold tuning performed.
4. Final threshold selected = 0.95.
5. Model and artifacts saved for deployment.
6. System ready for Flask integration.